# Re-Ranking
In the context of RAG (Retrieval-Augmented Generation), reranking of retrieval results is a crucial step that refines the initial set of retrieved documents based on their relevance to the input query. This process involves re-scoring the retrieved documents using a more sophisticated model, such as a cross-encoder, to better capture the semantic similarity between the query and the documents. The reranked list of documents is then used as input for the generation model, ensuring that the most relevant and accurate information is utilized to generate the final output.

Here are the steps:

- Loading the reranking model
- Lading retrieval results
- Calculating reranking score
- Generating a reply on the reranked documents


#### Visual Improvements

In [1]:
from rich.console import Console
from rich_theme_manager import Theme, ThemeManager
import pathlib

theme_dir = pathlib.Path("themes")
theme_manager = ThemeManager(theme_dir=theme_dir)
dark = theme_manager.get("dark")

# Create a console with the dark theme
console = Console(theme=dark)
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

## Loading the Reranking Model

In [2]:
from sentence_transformers import CrossEncoder 
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
console.print(cross_encoder.model)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 384, padding_idx=0)
      (position_embeddings): Embedding(512, 384)
      (token_type_embeddings): Embedding(2, 384)
      (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-5): 6 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
              (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
          )
          (intermediate): BertIntermediate(
            (dense): Linear(in_features=384, out_features=1536, bias=True)
            (intermediate_act_fn): GELUActivation()
          )
          (output): BertOutput(
            (dense): Linear(in_features=1536, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
      )
    )
    (pooler): BertPooler(
      (dense): Linear(in_features=384, out_features=384, bias=True)
      (activation): Tanh()
    )
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (classifier): Linear(in_features=384, out_features=1, bias=True)
)

## Loading retrieval results
We will load the retrieval results from the previous Hybrid-Search notebook, to avoid repetition. We can ignore the scores of the dense and sparse index, as we will calculate the ranking score based on the text of the document/chunk.

In [3]:
import json
hybrid_search_results = {}
with open('data/dense_results.json') as f:
    dense_results = json.load(f)
    for doc in dense_results:
        hybrid_search_results[doc['id']] = doc
with open('data/sparse_results.json') as f:
    sparse_results = json.load(f)
    for doc in sparse_results:
        hybrid_search_results[doc['id']] = doc
console.print(hybrid_search_results)

{
    19: {
        'id': 19,
        'text': '3.8 â Mixtral_8x7B 3.5 32 > $3.0 i] 228 fos a 2.0 0 5k 10k 15k 20k 25k 30k Context length Passkey 
Performance ry 3.8 â Mixtral_8x7B 3.5 0.8 32 > 0.6 $3.0 i] 228 04 fos 0.2 a 2.0 0.0 OK 4K 8K 12K 16K 20K 24K 28K 0 
5k 10k 15k 20k 25k 30k Seq Len Context length Figure 4: Long range performance of Mixtral. (Left) Mixtral has 100% 
retrieval accuracy of the Passkey task regardless of the location of the passkey and length of the input sequence. 
(Right) The perplexity of Mixtral on the proof-pile dataset decreases monotonically as the context length 
increases.\n\nThis chunk is part of the "Results" section of the document, specifically discussing the long-range 
performance of the Mixtral 8x7B model on a passkey retrieval task and its perplexity on the proof-pile dataset, 
highlighting its effectiveness in handling long contexts.',
        'metadata': {'title': 'Mixtral of Experts', 'arxiv_id': '2401.04088', 'references': ['1905.07830']}
    },
    48: {
        'id': 48,
        'text': '13\n\nThe chunk appears to be part of the detailed results section of the document, specifically 
discussing the performance metrics and comparisons of the Mixtral 8x7B model against other models like Llama 2 and 
GPT-3.5 across various benchmarks, highlighting its efficiency and effectiveness in different tasks.',
        'metadata': {'title': 'Mixtral of Experts', 'arxiv_id': '2401.04088', 'references': ['1905.07830']}
    },
    4: {
        'id': 4,
        'text': 'Instruct under the Apache 2.0 license1, free for academic and commercial usage, ensuring broad 
accessibility and potential for diverse applications. To enable the community to run Mixtral with a fully 
open-source stack, we submitted changes to the vLLM project, which integrates Megablocks CUDA kernels for efficient
inference. Skypilot also allows the deployment of vLLM endpoints on any instance in the cloud. # 2 Architectural 
details Mixtral is based on a transformer architecture [31] and uses the same modifications as described in [18], 
with the notable exceptions that Mix- tral supports a fully dense context length of 32k tokens, and the feed- 
forward blocks are replaced by Mixture-of-Expert layers (Section 2.1). The model architecture parameters are 
summarized in Table 1.\n\nThis chunk is part of the "Introduction" and "Architectural details" sections of the 
document, which discusses the Mixtral 8x7B model, its licensing, accessibility, and the architectural modifications
made to the transformer framework, including the implementation of Mixture-of-Expert layers and context length 
specifications.',
        'metadata': {'title': 'Mixtral of Experts', 'arxiv_id': '2401.04088', 'references': ['1905.07830']}
    },
    10: {
        'id': 10,
        'text': '¢ Math: GSM8K [9] (8-shot) with maj@8 and MATH [17] (4-shot) with maj@4 â ¢ Code: Humaneval [4] 
(0-shot) and MBPP [1] (3-shot) â ¢ Popular aggregated results: MMLU [16] (5-shot), BBH [29] (3-shot), and AGI Eval 
[34] (3-5-shot, English multiple-choice questions only)\n\nThis chunk is part of the "Results" section of the 
document, where the authors compare the performance of the Mixtral model against other models across various 
benchmarks, specifically highlighting its capabilities in mathematics and code generation tasks.',
        'metadata': {'title': 'Mixtral of Experts', 'arxiv_id': '2401.04088', 'references': ['1905.07830']}
    },
    2: {
        'id': 2,
        'text': 'expertsâ ) to process the token and combine their output additively. This technique increases the 
number of parameters of a model while controlling cost and latency, as the model only uses a fraction of the total 
set of parameters per token. Mixtral is pretrained with multilingual data using a context size of 32k tokens. It 
either matches or exceeds the performance of Llama 2 70B and GPT-3.5, over several benchmarks. In particular, 
Mixture of Experts Layer i gating inputs af outputs router

In [4]:
#  This is the query that we used for the retrieval of the above documents
query = "What is context size of Mixtral?"

## Calculating the re-ranking scores
We are using the cross_encoder to calculate the match score.

In [5]:
pairs = [[query, doc['text']] for doc in hybrid_search_results.values()] 
scores = cross_encoder.predict(pairs) 

console.print(scores)

[  5.208254   -1.3020442   2.9193153  -3.6383343   6.870935    3.8961406
  -3.5867338  -0.2788083  -0.640713   -3.918332    1.8424054   2.3611677
   3.0408275 -11.29175   -11.221519 ]

### Selecting top 3 reranked documents

In [6]:
# Combine scores with corresponding document IDs
results_with_scores = [
    (doc_id, hybrid_search_results[doc_id]['text'], score)
    for doc_id, score in zip(hybrid_search_results.keys(), scores)
]

# Sort results by score in descending order and take the top 3
top_results = sorted(results_with_scores, key=lambda x: x[2], reverse=True)[:3]
import numpy as np
from rich.table import Table
table = Table(title="Top 3 Documents after Reranking", show_lines=True)

table.add_column("ID", justify="right", style="cyan", no_wrap=True)
table.add_column("Score", justify="right", style="green", no_wrap=True)
table.add_column("Document", style="#e87d3e")

# Add rows to the table with top 3 results
for doc_id, text, score in top_results:
    table.add_row(str(doc_id), f"{score:.4f}", text)

console.print(table)

                                          Top 3 Documents after Reranking                                          
┏━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ ID ┃  Score ┃ Document                                                                                          ┃
┡━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  2 │ 6.8709 │ expertsâ ) to process the token and combine their output additively. This technique increases the │
│    │        │ number of parameters of a model while controlling cost and latency, as the model only uses a      │
│    │        │ fraction of the total set of parameters per token. Mixtral is pretrained with multilingual data   │
│    │        │ using a context size of 32k tokens. It either matches or exceeds the performance of Llama 2 70B   │
│    │        │ and GPT-3.5, over several benchmarks. In particular, Mixture of Experts Layer i gating inputs af  │
│    │        │ outputs router expert                                                                             │
│    │        │                                                                                                   │
│    │        │ This chunk is part of the section discussing the architecture and functionality of the Mixtral    │
│    │        │ 8x7B model, specifically focusing on the Mixture of Experts (MoE) mechanism, which allows the     │
│    │        │ model to utilize a subset of its parameters for each token processed, enhancing efficiency and    │
│    │        │ performance across various benchmarks.                                                            │
├────┼────────┼───────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 19 │ 5.2083 │ 3.8 â Mixtral_8x7B 3.5 32 > $3.0 i] 228 fos a 2.0 0 5k 10k 15k 20k 25k 30k Context length Passkey │
│    │        │ Performance ry 3.8 â Mixtral_8x7B 3.5 0.8 32 > 0.6 $3.0 i] 228 04 fos 0.2 a 2.0 0.0 OK 4K 8K 12K  │
│    │        │ 16K 20K 24K 28K 0 5k 10k 15k 20k 25k 30k Seq Len Context length Figure 4: Long range performance  │
│    │        │ of Mixtral. (Left) Mixtral has 100% retrieval accuracy of the Passkey task regardless of the      │
│    │        │ location of the passkey and length of the input sequence. (Right) The perplexity of Mixtral on    │
│    │        │ the proof-pile dataset decreases monotonically as the context length increases.                   │
│    │        │                                                                                                   │
│    │        │ This chunk is part of the "Results" section of the document, specifically discussing the          │
│    │        │ long-range performance of the Mixtral 8x7B model on a passkey retrieval task and its perplexity   │
│    │        │ on the proof-pile dataset, highlighting its effectiveness in handling long contexts.              │
├────┼────────┼───────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 18 │ 3.8961 │ Table 4: Comparison of Mixtral with Llama on Multilingual Benchmarks. On ARC Challenge,           │
│    │        │ Hellaswag, and MMLU, Mixtral outperforms Llama 2 70B on 4 languages: French, German, Spanish, and │
│    │        │ Italian. # 3.2 Long range performance To assess the capabilities of Mixtral to tackle long        │
│    │        │ context, we evaluate it on the passkey retrieval task introduced in [23], a synthetic task        │
│    │        │ designed to measure the ability of the model to retrieve a passkey inserted randomly in a long    │
│    │        │ prompt. Results in Figure 4 (Left) show that Mixtral achieves a 100% retrieval accuracy           │
│    │        │ regardless of the context length or the position of passkey in the sequence. Figure 4 (Right)     │
│    │        │ shows that the perplexity of Mixtral on 

## Using merged results to generate a reply
We can now take the improved merged results and call the LLM to generate the reply to the user's query.

In [7]:
# define a variable to hold the search results for the generation model
search_results = [doc[1] for doc in top_results]

In [8]:
from dotenv import load_dotenv

load_dotenv()


True

In [9]:
# Now time to connect to the large language model
from openai import OpenAI
from rich.text import Text

client = OpenAI()
completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "You are chatbot, an research expert. Your top priority is to help guide users to understand reserach papers."},
        {"role": "user", "content": query},
        {"role": "assistant", "content": str(search_results)}
    ]
)

response_text = Text(completion.choices[0].message.content)

In [10]:
from rich.panel import Panel

panel = Panel(response_text, title=f"Hybrid Search with Reranking Reply to \"{query}\"")
console.print(panel)

╭─────────────────── Hybrid Search with Reranking Reply to "What is context size of Mixtral?" ────────────────────╮
│ Mixtral is pretrained with multilingual data using a context size of 32k tokens.                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯